# Consultas en MySQL


In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import OperationalError

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "notebooks" / "query_worker.ipynb").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. Abre y ejecuta este notebook desde el proyecto."
    )

ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)

MYSQL_HOST = os.getenv("MYSQL_HOST", "127.0.0.1")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "chobani_dw")

if not MYSQL_USER or not MYSQL_PASSWORD:
    raise ValueError(f"Define MYSQL_USER y MYSQL_PASSWORD en {ENV_PATH}.")

database_url = URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE,
    query={"charset": "utf8mb4"},
)
engine = create_engine(database_url, pool_pre_ping=True)
print(f"Configuración lista para {MYSQL_DATABASE} en {MYSQL_HOST}:{MYSQL_PORT}.")

Configuración lista para chobani_dw en 127.0.0.1:3306.


In [2]:
try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
except OperationalError as exc:
    error_code = getattr(exc.orig, "args", (None,))[0]
    if error_code == 1045:
        raise RuntimeError(
            f"MySQL rechazó la autenticación para {MYSQL_USER!r}. "
            f"Verifica usuario y contraseña en {ENV_PATH}; no compartas la contraseña."
        ) from exc
    if error_code == 1049:
        raise RuntimeError(
            f"La base {MYSQL_DATABASE!r} no existe o no es accesible para este usuario."
        ) from exc
    raise

print("Conexión a MySQL verificada.")

Conexión a MySQL verificada.


## Consulta


In [74]:
def run_query(query: str, params: dict | None = None) -> pd.DataFrame | None:
    """Ejecuta una sentencia y devuelve filas, si la sentencia las produce."""
    with engine.begin() as connection:
        result = connection.execute(text(query), params or {})
        if result.returns_rows:
            return pd.DataFrame(result.fetchall(), columns=result.keys())
        print(f"Sentencia ejecutada; filas afectadas: {result.rowcount}.")
        return None


CSV_OUTPUT_DIR = PROJECT_ROOT / "data" / "query_exports"
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def export_query_to_csv(result: pd.DataFrame | None, filename: str) -> Path:
    """Exporta a CSV el resultado tabular de una consulta."""
    if result is None:
        raise ValueError("La consulta no devolvió filas para exportar.")

    output_path = CSV_OUTPUT_DIR / filename
    result.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"CSV exportado: {output_path}")
    return output_path


SQL_QUERY_1 = """
    SELECT region AS `Región`, 
        channel AS `Canal de venta`,
        SUM(`total_from_lines`) AS `Ventas totales de 2024`,
        COUNT(DISTINCT order_id) AS `Número de pedidos`, 
        SUM(`total_from_lines`)/COUNT(DISTINCT order_id) AS `Ticket promedio`
    FROM `fact_pedidos` a 
    JOIN `dim_clientes` b
    ON a.customer_id = b.customer_id
    WHERE `status` = "Entregado"
    AND YEAR(order_date) = 2024
    GROUP BY region, channel
    ORDER BY `Ventas totales de 2024`

"""
query_result_1 = run_query(SQL_QUERY_1)
export_query_to_csv(query_result_1, "query_1_sales_by_region_2024.csv")
query_result_1

CSV exportado: /Users/orozcos/Desktop/Tests/Chobani Test/Take-Home-Test-Nov-26/data/query_exports/query_1_sales_by_region_2024.csv


,Región,Canal de venta,Ventas totales de 2024,Número de pedidos,Ticket promedio
0,No Especificada,Mayorista,103013.52,15,6867.568000
1,No Especificada,Foodservice,325606.91,52,6261.671346
2,Centro,Mayorista,375004.39,65,5769.298308
3,Norte,Foodservice,460061.47,71,6479.739014
4,Sur,Autoservicio,539016.42,81,6654.523704
5,Occidente,Foodservice,724058.30,113,6407.595575
6,Sureste,Foodservice,746634.02,114,6549.421228
7,Sureste,Conveniencia,748975.28,127,5897.443150
8,Sureste,Autoservicio,749418.51,121,6193.541405
9,Occidente,Mayorista,790377.32,116,6813.597586


In [78]:

SQL_QUERY_2 = """WITH CUSTOMER_TOTAL AS (
    SELECT 
        a.customer_id,
        customer_name,
        product_name,
        category, 
        SUM(line_amount) AS customer_product_total,
        SUM(SUM(line_amount)) OVER (PARTITION BY a.customer_id) AS customer_total
    FROM `dim_clientes` a
    JOIN `fact_pedidos` b       ON a.customer_id = b.customer_id
    JOIN `fact_lineas_pedido` c ON c.order_id = b.order_id
    JOIN `dim_productos` d      ON c.product_id = d.product_id
    WHERE b.status = 'Entregado'
    GROUP BY a.customer_id, d.product_id, customer_name, product_name, category
),
RANKED AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id 
            ORDER BY customer_product_total DESC, 
                     product_name ASC, 
                     category ASC
        ) AS ranking,
        (customer_product_total/ customer_total) * 100 AS pct_cliente
    FROM CUSTOMER_TOTAL
)
SELECT 
    customer_id,
    customer_name,
    ranking,
    product_name,
    category, 
    customer_product_total,
    pct_cliente
FROM RANKED
WHERE ranking <= 3
ORDER BY customer_id, ranking

"""
query_result_2 = run_query(SQL_QUERY_2)
export_query_to_csv(query_result_2, "query_2_top_products_by_customer.csv")
query_result_2

CSV exportado: /Users/orozcos/Desktop/Tests/Chobani Test/Take-Home-Test-Nov-26/data/query_exports/query_2_top_products_by_customer.csv


,customer_id,customer_name,ranking,product_name,category,customer_product_total,pct_cliente
0,C0001,Distribuidora Premier 1,1,Creamer Avellana,Creamer,11842.32,11.752241
1,C0001,Distribuidora Premier 1,2,Otros Requeson,Otros,9350.52,9.279395
2,C0001,Distribuidora Premier 1,3,Creamer Coco,Creamer,7815.04,7.755595
3,C0002,Distribuidora Unidos 2,1,Creamer Avellana,Creamer,14647.08,7.855053
4,C0002,Distribuidora Unidos 2,2,Creamer Vainilla,Creamer,13659.50,7.325425
...,...,...,...,...,...,...,...
595,C0199,Distribuidora Unidos 199,2,Queso Crema Original,Queso Crema,10762.30,6.891238
596,C0199,Distribuidora Unidos 199,3,Otros Kefir,Otros,9865.52,6.317019
597,C0200,Distribuidora Alianza 200,1,Queso Crema Original,Queso Crema,13219.24,8.411476
598,C0200,Distribuidora Alianza 200,2,Creamer Avellana,Creamer,12387.69,7.882357


In [81]:

SQL_QUERY_3 = """WITH RECURSIVE months AS (
    -- Genera el primer día de cada mes del rango de interés
    SELECT DATE('2024-01-01') AS month_start
    UNION ALL
    SELECT DATE_ADD(month_start, INTERVAL 1 MONTH)
    FROM months
    WHERE month_start < DATE('2024-12-01')
),
regions AS (
    -- Universo de regiones: todas las que existen en el negocio
    SELECT DISTINCT region FROM dim_clientes
),
spine AS (
    -- Producto cartesiano regiones × meses = calendario completo
    SELECT r.region, m.month_start,
           YEAR(m.month_start)  AS periodo,
           MONTH(m.month_start) AS mes
    FROM regions r
    CROSS JOIN months m
),
sales AS (
    -- Ventas reales por región y mes
    SELECT 
        b.region,
        DATE_FORMAT(a.order_date, '%Y-%m-01') AS month_start,
        SUM(a.total_from_lines)               AS total_ventas
    FROM fact_pedidos a
    JOIN dim_clientes b ON a.customer_id = b.customer_id
    WHERE a.status = 'Entregado'
    GROUP BY b.region, DATE_FORMAT(a.order_date, '%Y-%m-01')
),
combined AS (
    -- LEFT JOIN + COALESCE: meses sin ventas se materializan con 0
    SELECT 
        s.region, s.periodo, s.mes,
        COALESCE(sl.total_ventas, 0) AS total_ventas
    FROM spine s
    LEFT JOIN sales sl
        ON sl.region = s.region
       AND sl.month_start = s.month_start
),
with_lag AS (
    SELECT 
        *,
        LAG(total_ventas) OVER (
            PARTITION BY region 
            ORDER BY periodo, mes
        ) AS total_ventas_mes_anterior,
        AVG(total_ventas) OVER (
            PARTITION BY region 
            ORDER BY periodo, mes 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS promedio_ventas_3_meses
    FROM combined
),
with_var AS (
    SELECT 
        *,
        (total_ventas - total_ventas_mes_anterior) 
            / NULLIF(total_ventas_mes_anterior, 0) * 100 
            AS variacion_porcentual
    FROM with_lag
)
SELECT 
    *,
    CASE 
        WHEN variacion_porcentual < 15 THEN 'Caída Crítica' 
        ELSE '' 
    END AS Bandera_Caida
FROM with_var
ORDER BY periodo DESC, mes DESC, region;
"""
query_result_3 = run_query(SQL_QUERY_3)
export_query_to_csv(query_result_3, "query_3_monthly_sales_variation.csv")
query_result_3

CSV exportado: /Users/orozcos/Desktop/Tests/Chobani Test/Take-Home-Test-Nov-26/data/query_exports/query_3_monthly_sales_variation.csv


,region,periodo,mes,total_ventas,total_ventas_mes_anterior,promedio_ventas_3_meses,variacion_porcentual,Bandera_Caida
0,Centro,2024,12,238017.97,279623.99,253651.950000,-14.879274,Caída Crítica
1,No Especificada,2024,12,66765.24,27516.81,44437.210000,142.634375,
2,Norte,2024,12,272729.00,316866.73,278177.943333,-13.929430,Caída Crítica
3,Occidente,2024,12,250785.46,239654.39,266956.733333,4.644634,Caída Crítica
4,Sur,2024,12,374248.05,336592.33,346068.276667,11.187338,Caída Crítica
...,...,...,...,...,...,...,...,...
67,No Especificada,2024,1,14799.84,None,14799.840000,None,
68,Norte,2024,1,163182.62,None,163182.620000,None,
69,Occidente,2024,1,239767.72,None,239767.720000,None,
70,Sur,2024,1,177915.62,None,177915.620000,None,
